In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## BID system calculation

In [2]:
import numpy as np


ambulance_pos = (36.80, 10.18)
current_time = 0  # in minutes
tau = 20 

# Hospital data: [lat, lon, energy (capacity), load, estimated_time, patient_severity]
hospitals = [
    {"id": "H1", "pos": (36.82, 10.20), "capacity": 0.8, "specialization_match": 1, "estimated_time": 6, "patient_severity": 0.9},
    {"id": "H2", "pos": (36.75, 10.25), "capacity": 0.6, "specialization_match": 1, "estimated_time": 15, "patient_severity": 0.9},
    {"id": "H3", "pos": (36.78, 10.10), "capacity": 0.9, "specialization_match": 0, "estimated_time": 10, "patient_severity": 0.9}
]


alpha = 0.5  # weight for distance
beta = 0.2   # weight for energy / capacity
gamma = 0.3  # weight for load / specialization
epsilon = 0.1  # small constant to avoid div by zero

  
def euclidean_distance(a, b):
    return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

# Max values for normalization
Emax = max([h["capacity"] for h in hospitals])
Lmax = max([h["specialization_match"] for h in hospitals])


for h in hospitals:
    di_j = euclidean_distance(ambulance_pos, h["pos"])  # could replace with A* distance
    Ei = h["capacity"]
    Li = h["specialization_match"]
    Tj = h["estimated_time"]
    

    I = 1 if (Tj - current_time <= tau) else 0
    

    bid = alpha * (1 / (di_j + epsilon)) + beta * (Ei / Emax) + gamma * (Li / Lmax) * I
    h["bid"] = bid


winner = max(hospitals, key=lambda x: x["bid"])
print("Hospital bids:")
for h in hospitals:
    print(f"{h['id']}: bid={h['bid']:.3f}")
print(f"\nSelected hospital: {winner['id']}")

Hospital bids:
H1: bid=4.375
H2: bid=3.121
H3: bid=2.940

Selected hospital: H1
